# Body keypoint triangulation (MediaPipe Pose, 33 landmarks)

Pure-geometric companion to `ransac_keypoint_reconstruction.ipynb`, sharing the same
`v3` calibration/triangulation/smoothing/visualization package but for the 33-landmark
MediaPipe Pose **body** keypoints (single subject, single instance per frame -- no
detection/tracking/handedness step is needed, since `h5_explore.ipynb` already
extracted one body-landmark set per camera/frame into `extracted_2d_keypoints.json`).

1. Build per-camera/frame keypoint arrays directly from `extracted_2d_keypoints.json`
   (`v3.keypoints_io.build_keypoint_arrays`).
2. Triangulate each of the 33 landmarks into 3D via weighted multi-view DLT, with a
   RANSAC step (exhaustive camera pairs, reprojection-error inlier counting on the
   hip-center) over all 128 frames.
3. Kalman/RTS-smooth, jitter-flag, and Savitzky-Golay-smooth + 2x-upsample the
   trajectories (same `v3.smoothing` pipeline as the hand notebook).
4. Visualize the resulting 3D body skeleton in viser.

Saves `body_keypoint_3d.npz`, consumed by `smplx_body_fit.ipynb` (Phase 3).

Generated by `v3/notebooks/build_body_notebook.py`, which is the source of truth --
edit that script and re-run `python -m v3.notebooks.build_body_notebook`, don't edit
this notebook directly. Shared calibration/triangulation/smoothing/visualization logic
lives in the `v3` package (also used by `ransac_keypoint_reconstruction.ipynb`).

Runs entirely in `oak_env`.

In [ ]:
import time

import numpy as np

from v3.calibration import load_calibration
from v3.keypoints_io import load_extracted_keypoints, build_keypoint_arrays
from v3.triangulation import undistort_keypoints, ransac_triangulate_sequence
from v3.smoothing import smooth_sequence, jitter_flags, savgol_and_upsample_2x
from v3.skeletons import BODY_CONNECTIONS
from v3 import visualization as viz

H5_PATH = '/data/fmalmb/DATA/tmp/Testdata.h5'

In [ ]:
CAMERA_CALIB, CAMERA_WORLD_TRANSFORMS, PROJECTION_MATRICES, camera_ids = load_calibration(H5_PATH)
N_CAMS = len(camera_ids)

In [ ]:
keypoints_2d = load_extracted_keypoints('extracted_2d_keypoints.json')

In [ ]:
# --- Stage 1: build per-camera/frame keypoint arrays from extracted_2d_keypoints.json --
# Body keypoints were already extracted by h5_explore.ipynb's MediaPipe Pose pass -- no
# YOLO/tracking/handedness step is needed (single subject, single 33-landmark set per
# camera/frame).
FRAME_OFFSET = 0
N_FRAMES = 128
N_JOINTS = 33

frame_keys = [f'{i:06d}' for i in range(FRAME_OFFSET, FRAME_OFFSET + N_FRAMES)]
keypoints_px, keypoints_conf, presence = build_keypoint_arrays(
    keypoints_2d, camera_ids, frame_keys, field='body', n_joints=N_JOINTS)

for ci, cam_id in enumerate(camera_ids):
    n_present = int(presence[ci].sum())
    print(f'{cam_id}: {n_present}/{N_FRAMES} frames with body landmarks')

In [ ]:
# --- Stage 2: RANSAC multi-view triangulation ----------------------------------------
RANSAC_REPROJ_THRESH_PX = 50.0  # same threshold as the hand pipeline; hip-center seed
                                 # residuals (printed below) sit comfortably under it,
                                 # same as the hands' wrist residuals.

keypoints_px_undist = undistort_keypoints(keypoints_px, CAMERA_CALIB, camera_ids)

result = ransac_triangulate_sequence(
    keypoints_px_undist, keypoints_conf, presence, PROJECTION_MATRICES, camera_ids,
    seed_joint_idx=(23, 24),  # hip-center (left_hip=23, right_hip=24) -- stable, central,
                               # least-occluded body landmark, analogous to SMPL-X's pelvis.
    occlusion=None,
    reproj_thresh_px=RANSAC_REPROJ_THRESH_PX,
)
joints3d = result['joints3d']
valid = result['valid']
n_inliers = result['n_inliers']
reproj_err = result['reproj_err']
hip_residuals = result['seed_residuals']
hysteresis_overrides = result['hysteresis_overrides']
n_valid_frames = result['n_valid_frames']

n_valid = int(valid.sum())
print(f'body: {n_valid}/{N_FRAMES} frames valid')
ni = n_inliers[valid]
print(f'  inlier cams: min={ni.min()}, mean={ni.mean():.2f}, max={ni.max()}')
print(f'  mean reprojection error: {np.nanmean(reproj_err[valid]):.1f}px')
print(f'  hysteresis kept previous camera set: {int(hysteresis_overrides)}/{int(n_valid_frames)} valid frames')
print(f'hip-center RANSAC residual (px): mean={hip_residuals.mean():.1f}, '
      f'median={np.median(hip_residuals):.1f}, max={hip_residuals.max():.1f}')

In [ ]:
# --- Stage 2c: Kalman/RTS smoothing of joint trajectories -----------------------------
KALMAN_PROCESS_STD = 0.03     # m / frame^2 acceleration-noise std (raise if smoothing lags fast motion)
KALMAN_MEAS_STD_BASE = 0.01   # m position-measurement std for an n_inliers>=3 triangulation
KALMAN_MEAS_STD_2CAM = 0.015  # m position-measurement std for an n_inliers==2 triangulation
MIN_INLIERS_BASE = 3          # n_inliers is in {2, 3} for body, same range as the hands

joints3d_smooth, valid_smooth = smooth_sequence(
    joints3d, valid, n_inliers,
    process_std=KALMAN_PROCESS_STD,
    meas_std_base=KALMAN_MEAS_STD_BASE,
    meas_std_2cam=KALMAN_MEAS_STD_2CAM,
    min_inliers_base=MIN_INLIERS_BASE,
)

hip_raw = (joints3d[:, 23] + joints3d[:, 24]) / 2.0
hip_smooth = (joints3d_smooth[:, 23] + joints3d_smooth[:, 24]) / 2.0
delta = np.linalg.norm(hip_raw[valid] - hip_smooth[valid], axis=-1)
print(f'hip-center: raw-vs-smoothed correction (cm): '
      f'mean={delta.mean() * 100:.2f}, median={np.median(delta) * 100:.2f}, max={delta.max() * 100:.2f}')

In [ ]:
# --- Stage 2d: jitter flagging (raw RANSAC vs Kalman/RTS-smoothed hip-center) ---------
JITTER_MAD_FACTOR = 4.0  # flag residuals > median + this many MADs above it

jitter_flag, hip_jitter_residual = jitter_flags(
    joints3d, joints3d_smooth, valid, seed_joint_idx=(23, 24), mad_factor=JITTER_MAD_FACTOR)

valid_idx = np.where(valid)[0]
res = hip_jitter_residual[valid_idx]
med = np.median(res)
mad = np.median(np.abs(res - med)) + 1e-9
thresh = med + JITTER_MAD_FACTOR * mad
flagged = valid_idx[res > thresh]

print(f'hip-center raw-vs-smoothed residual (cm): median={med * 100:.2f}, '
      f'mad={mad * 100:.2f}, threshold={thresh * 100:.2f}, max={res.max() * 100:.2f}')
print(f'flagged frames ({len(flagged)}/{len(valid_idx)}): {(flagged + FRAME_OFFSET).tolist()}')

In [ ]:
# --- Stage 2e: Savitzky-Golay smoothing + 2x frame-rate interpolation -----------------
SG_WINDOW = 7  # frames (must be odd and <= the valid span length)
SG_POLYORDER = 3

sg_result = savgol_and_upsample_2x(joints3d_smooth, valid_smooth, sg_window=SG_WINDOW, sg_polyorder=SG_POLYORDER)
joints3d_sg = sg_result['joints3d_sg']
joints3d_2x = sg_result['joints3d_2x']
valid_2x = sg_result['valid_2x']
N_FRAMES_2X = sg_result['n_frames_2x']
nearest_orig_2x = sg_result['nearest_orig_2x']

# Nearest-original-frame upsampling of per-frame diagnostics, for visualization at 2x rate.
jitter_flag_2x = jitter_flag[nearest_orig_2x]
n_inliers_2x = n_inliers[nearest_orig_2x]

hip_smooth = (joints3d_smooth[:, 23] + joints3d_smooth[:, 24]) / 2.0
hip_sg = (joints3d_sg[:, 23] + joints3d_sg[:, 24]) / 2.0
delta = np.linalg.norm(hip_smooth[valid_smooth] - hip_sg[valid_smooth], axis=-1)
print(f'hip-center: Savitzky-Golay correction (cm): mean={delta.mean() * 100:.3f}, '
      f'max={delta.max() * 100:.3f}; {int(valid_2x.sum())}/{N_FRAMES_2X} frames valid at 2x rate')

np.savez_compressed(
    'body_keypoint_3d.npz',
    joints3d=joints3d, valid=valid, n_inliers=n_inliers, reproj_err=reproj_err,
    joints3d_smooth=joints3d_smooth, valid_smooth=valid_smooth,
    joints3d_sg=joints3d_sg, joints3d_2x=joints3d_2x, valid_2x=valid_2x,
    n_inliers_2x=n_inliers_2x, jitter_flag=jitter_flag, jitter_flag_2x=jitter_flag_2x,
    nearest_orig_2x=nearest_orig_2x, n_frames_2x=np.array(N_FRAMES_2X),
    camera_ids=np.array(camera_ids), frame_indices=np.arange(FRAME_OFFSET, FRAME_OFFSET + N_FRAMES),
)
print('Saved body_keypoint_3d.npz')

In [ ]:
# --- Stage 3: viser visualization -----------------------------------------------------
import viser

BODY_COLOR = (180, 220, 255)
LOW_CONFIDENCE_COLOR = (140, 140, 140)
JITTER_COLOR = (230, 40, 40)  # frames where the raw RANSAC hip-center deviated sharply from
                               # the Kalman/RTS-smoothed trajectory (jitter_flag above)

# Origin: centroid of all valid joints in the first valid frame.
ORIGIN_WORLD = viz.compute_origin(joints3d, valid)


def main():
    server = viser.ViserServer()
    running = True
    try:
        server.scene.set_up_direction('+z')
        server.gui.configure_theme(dark_mode=True)
        viz.setup_cameras(server, CAMERA_CALIB, CAMERA_WORLD_TRANSFORMS, ORIGIN_WORLD)

        server.initial_camera.position = (2.5, 2.0, 1.6)
        server.initial_camera.look_at = (0.0, 0.0, 0.9)
        server.initial_camera.up = (0.0, 0.0, 1.0)

        print('Viser: open the URL shown above')
        print(f'Origin offset (calibration world): {ORIGIN_WORLD}')

        stop_button = server.gui.add_button('Stop viewer')
        frame_slider = server.gui.add_slider('Frame (2x rate)', min=0, max=N_FRAMES_2X - 1, step=1, initial_value=0)
        play_checkbox = server.gui.add_checkbox('Play (loop)', initial_value=True)
        fps_slider = server.gui.add_slider('Playback FPS', min=1, max=60, step=1, initial_value=20)

        @stop_button.on_click
        def _stop(_) -> None:
            nonlocal running
            running = False

        last_advance = time.time()

        while running:
            fi = frame_slider.value
            name = '/body'
            if valid_2x[fi]:
                if jitter_flag_2x[fi]:
                    color = JITTER_COLOR
                elif n_inliers_2x[fi] >= 3:
                    color = BODY_COLOR
                else:
                    color = LOW_CONFIDENCE_COLOR
                viz.add_skeleton(server, name, joints3d_2x[fi], BODY_CONNECTIONS, color,
                                  origin_world=ORIGIN_WORLD)
            else:
                viz.add_empty_skeleton(server, name)

            if play_checkbox.value:
                now = time.time()
                if now - last_advance >= 1.0 / fps_slider.value:
                    frame_slider.value = (fi + 1) % N_FRAMES_2X
                    last_advance = now

            time.sleep(0.02)
    except KeyboardInterrupt:
        print('Viewer interrupted.')
    finally:
        server.stop()
        print('Viser stopped.')


main()